# LSTM Model Training

In [1]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [3]:
X = np.load('../data/processed/X_sequences.npy')
y = np.load('../data/processed/y_sequences.npy')

# Stratified split to ensure failures appear in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')
print(f'Train failures: {int(y_train.sum())}, Test failures: {int(y_test.sum())}')

Train size: 23976, Test size: 5994
Train failures: 4189, Test failures: 1047


In [4]:
# Failure Probability = 1 / (1 + e^(-z)) — sigmoid for binary classification

# Class weights to fix recall=0 problem (imbalanced dataset)
class_weights_arr = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: float(class_weights_arr[0]), 1: float(class_weights_arr[1])}
print(f'Class weights: {class_weight_dict}')

model = keras.Sequential([
    keras.Input(shape=(X_train.shape[1], X_train.shape[2])),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(32),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=[
        keras.metrics.AUC(name='auc'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.Precision(name='precision'),
        'accuracy'
    ]
)
model.summary()

Class weights: {0: 0.6058523272855916, 1: 2.8617808546192407}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 30, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,881 (120.63 KB)

 Trainable params: 30,881 (120.63 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
import os
os.makedirs('../models', exist_ok=True)

callbacks = [
    EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True, mode='max', verbose=1),
    ModelCheckpoint('../models/motor_lstm_model.h5', save_best_only=True, monitor='val_auc', mode='max', verbose=1)
]

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
600/600 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4582 - auc: 0.4915 - loss: 0.6990 - precision: 0.1766 - recall: 0.5587
Epoch 1: val_auc improved from None to 0.51281, saving model to ../models/motor_lstm_model.h5


600/600 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.5845 - auc: 0.4981 - loss: 0.6924 - precision: 0.1763 - recall: 0.3796 - val_accuracy: 0.7375 - val_auc: 0.5128 - val_loss: 0.6846 - val_precision: 0.1894 - val_recall: 0.1420
Epoch 2/50
597/600 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6612 - auc: 0.5059 - loss: 0.6904 - precision: 0.1790 - recall: 0.2652
Epoch 2: val_auc did not improve from 0.51281
600/600 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.5796 - auc: 0.5021 - loss: 0.6918 - precision: 0.1742 - recall: 0.3802 - val_accuracy: 0.6639 - val_auc: 0.4976 - val_loss: 0.6877 - val_precision: 0.1665 - val_recall: 0.2189
Epoch 3/50
596/600 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6427 - auc: 0.5141 - loss: 0.6868 - precision: 0.1812 - recall: 0.3117
Epoch 3: val_auc did not improve from 0.51281
600/600 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - accuracy: 0.6198 - auc: 0.5115 - loss: 0.6915 - precision: 0.1792 - recall: 0.3324 - val_accuracy: 0.1879 - val_auc: 0.5015

In [ ]:
results = model.evaluate(X_test, y_test)
metric_names = ['loss', 'auc', 'recall', 'precision', 'accuracy']
for name, val in zip(metric_names, results):
    print(f'Test {name}: {val:.4f}')

print('Model saved to ../models/motor_lstm_model.h5')